# 04 — Đánh giá và giải thích

So sánh Dummy, Isolation Forest, One-Class SVM và Gradient Boosting trên **cùng một tập test**, với cấu hình và ngưỡng đã khóa từ notebook 02 và 03. Notebook chỉ đọc model đã lưu, không huấn luyện lại. Kết quả test không được dùng để chọn lại cấu hình hay ngưỡng.

Quy ước: `human = 0`, `bot = 1`; điểm càng lớn càng giống bot. Nếu thiếu file trong `models/`, chạy notebook 02 và 03 trước.

## Lịch sử sử dụng tập test

Tập test **không phải** hoàn toàn chưa được quan sát. Ghi lại để người đọc đánh giá đúng mức độ khách quan của kết quả:

| Ngày | Việc | Ảnh hưởng |
| --- | --- | --- |
| 29/10/2026 | Xem thống kê mô tả theo nhãn (trung vị, max) trên **toàn bộ** dữ liệu, gồm cả test, trước khi chia tập. Thống kê này dẫn đến việc dùng `log1p` trong pipeline One-Class SVM. | Có rò rỉ thông tin mô tả nhẹ từ test vào một quyết định tiền xử lý. Không có model nào được fit hay chọn trên test. |
| 31/10/2026 | Lần đánh giá đầu tiên trên test, chạy notebook này, sau khi cấu hình và ngưỡng đã được khóa trên validation (notebook 02, 03). | Không cấu hình hay ngưỡng nào được chọn lại sau khi xem kết quả test. |
| 09/11/2026 | Chạy lại notebook với cùng model đã lưu, để thêm cột truy vết vào `reports/metrics.csv`. | Model, cấu hình và ngưỡng không đổi; số liệu phải giống lần 31/10. |
| 10/11/2026 | Tích hợp và chạy lại notebook 01–04 từ code đã commit: Dummy/Gradient Boosting chính thức từ notebook 03 (lưới 6 cấu hình trên validation), anomaly từ notebook 02. | Đây là lần đánh giá test mới với bộ model chính thức. Không đổi cấu hình/ngưỡng dựa trên test. Model có hash nguồn/split, run_id và SHA-256 tệp; kết quả tạm trước đó được giữ trong `reports/history/2026-11-09_interim_*.csv`. |

Model Gradient Boosting và Dummy trong các lần chạy 31/10 và 09/11 là bản do Lê Hải Sơn huấn luyện tạm, trong khi chờ phần của Nguyễn Bá Thọ (notebook 03). Lần tích hợp model chính thức được ghi trong dòng 10/11 bên trên.

Các mốc 29/10, 31/10 và 09/11 ở trên được giữ theo lịch sử ghi trong commit `951a657`; không suy diễn thành một holdout chưa từng được xem. Con số 0,863 trước đây thuộc Gradient Boosting tạm (300 cây, max_depth=5), khác cấu hình chính thức hiện dùng.


In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
if not (project_root / "src").is_dir():
    raise RuntimeError("Mở notebook từ thư mục gốc dự án hoặc notebooks/.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import FEATURE_COLUMNS, RANDOM_STATE
print("Project root:", project_root)


Project root: C:\Users\Admin\.codex\worktrees\review-merge-dev\Social-Media-Bot-Fake-Account-Detection


In [2]:
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.inspection import permutation_importance
from sklearn.metrics import precision_recall_curve

from src.config import FIGURES_DIR, MODELS_DIR, REPORTS_DIR
from src.data import load_profiles, split_features
from src.features import build_features
from src.train import feature_matrix
from src.comparison import (
    DETECTORS, MODEL_NAMES, evaluate_models, file_sha256, load_bundles,
    model_scores as saved_model_scores, model_predictions as saved_model_predictions,
    validate_validation_metrics,
)

profiles, _ = load_profiles()
features = build_features(profiles)
train, validation, test = split_features(features)
X_val, y_val = validation[list(FEATURE_COLUMNS)], validation["label"].to_numpy()
X_test, y_test = test[list(FEATURE_COLUMNS)], test["label"].to_numpy()

# Refuse models from another source/split or library version before scoring test.
bundles = load_bundles()
display(pd.DataFrame({
    name: {
        "config": bundle["config"],
        "validation_f1_threshold": bundle["thresholds"].get("validation_f1"),
        "run_id": bundle["run_id"],
        "fit_seconds": round(bundle["fit_seconds"], 2),
    } for name, bundle in bundles.items()
}).T)


,config,validation_f1_threshold,run_id,fit_seconds
dummy,"{'constant': None, 'random_state': None, 'stra...",None,20261110T033850Z-7df46c79,0.03
isolation_forest,"{'n_estimators': 300, 'contamination': 'auto'}",-0.127698,20261110T033847Z-7441c005,0.4
one_class_svm,"{'nu': 0.05, 'gamma': 0.05}",-97.557444,20261110T033847Z-7441c005,0.78
gradient_boosting,"{'ccp_alpha': 0.0, 'criterion': 'deprecated', ...",0.369156,20261110T033850Z-7df46c79,4.33


## Các thiết lập được đánh giá

Mỗi mô hình được đánh giá ở ngưỡng mặc định và, trừ Dummy, ở ngưỡng đã chọn trên validation. Cùng một hàm chạy trên validation và test. Trên validation, AP được so với giá trị đã lưu cùng model để chắc chắn đang đọc đúng model.

In [3]:
import json


def model_scores(name, X):
    return saved_model_scores(name, bundles[name], X)


def model_predictions(name, X, threshold, scores):
    return saved_model_predictions(name, bundles[name], X, threshold, scores)


validation_results = evaluate_models(bundles, X_val, y_val, "validation")
validate_validation_metrics(bundles, validation_results)
validation_results.to_csv(REPORTS_DIR / "comparison_validation_metrics.csv", index=False)

# All model/config/threshold choices and validation checks precede test scoring.
metrics = evaluate_models(bundles, X_test, y_test, "test")
metrics.to_csv(REPORTS_DIR / "metrics.csv", index=False)
evaluation_run = {
    "evaluated_at_utc": metrics["evaluated_at_utc"].iloc[0],
    "evaluation_split": "test", "test_rows": len(test), "validation_rows": len(validation),
    "test_used_for_selection": False,
    "test_history": "See notebook 04: prior descriptive/test use and interim-model evaluations; "
                    "2026-11-10 integration evaluates the official notebook-03 models.",
    "metrics_sha256": file_sha256(REPORTS_DIR / "metrics.csv"),
    "code_sha256": {f"src/{name}.py": file_sha256(project_root / "src" / f"{name}.py")
                    for name in ("comparison", "evaluate", "train")},
    "models": {name: {
        "run_id": bundle["run_id"], "model_sha256": bundle["model_sha256"],
        "config": bundle["config"], "thresholds": bundle["thresholds"],
        "provenance": bundle["provenance"], "sklearn_version": bundle["sklearn_version"],
    } for name, bundle in bundles.items()},
}
(REPORTS_DIR / "evaluation_run.json").write_text(
    json.dumps(evaluation_run, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print("Đã lưu metrics và metadata vào reports/.")

bot_share = y_test.mean()
print(f"Mốc trên test: tỷ lệ bot {bot_share:.3f}; tất cả là bot cho F1 {2 * bot_share / (1 + bot_share):.3f}")
display(metrics[["model", "setting", "threshold", "precision_bot", "recall_bot", "f1_bot", "macro_f1",
                 "ap", "roc_auc", "accuracy", "flagged_share"]].round(3))
display(metrics[["model", "setting", "tn", "fp", "fn", "tp", "fit_seconds", "score_seconds"]].round(3))


Đã lưu metrics và metadata vào reports/.
Mốc trên test: tỷ lệ bot 0.332; tất cả là bot cho F1 0.498


,model,setting,threshold,precision_bot,recall_bot,f1_bot,macro_f1,ap,roc_auc,accuracy,flagged_share
0,dummy,default,NaN,0.000,0.000,0.000,0.401,0.332,0.500,0.668,0.000
1,isolation_forest,default,NaN,0.285,0.090,0.137,0.448,0.281,0.368,0.623,0.104
2,isolation_forest,validation_f1,-0.128,0.332,0.998,0.498,0.249,0.281,0.368,0.332,0.999
3,one_class_svm,default,NaN,0.361,0.054,0.094,0.440,0.373,0.564,0.654,0.050
4,one_class_svm,validation_f1,-97.557,0.348,0.864,0.496,0.403,0.373,0.564,0.418,0.824
5,gradient_boosting,default,NaN,0.756,0.633,0.689,0.776,0.789,0.861,0.810,0.278
6,gradient_boosting,validation_f1,0.369,0.679,0.733,0.705,0.775,0.789,0.861,0.797,0.358


,model,setting,tn,fp,fn,tp,fit_seconds,score_seconds
0,dummy,default,5003,0,2485,0,0.033,0.002
1,isolation_forest,default,4444,559,2262,223,0.403,0.082
2,isolation_forest,validation_f1,2,5001,4,2481,0.403,0.082
3,one_class_svm,default,4764,239,2350,135,0.782,0.305
4,one_class_svm,validation_f1,981,4022,339,2146,0.782,0.305
5,gradient_boosting,default,4496,507,913,1572,4.332,0.016
6,gradient_boosting,validation_f1,4144,859,664,1821,4.332,0.016


## So sánh validation và test

Cấu hình và ngưỡng được chọn trên validation, nên cần kiểm tra chúng có giữ được trên dữ liệu chưa dùng hay không.

In [4]:
keys = ["model", "setting"]
comparison = validation_results[keys + ["f1_bot", "macro_f1", "ap", "roc_auc"]].merge(
    metrics[keys + ["f1_bot", "macro_f1", "ap", "roc_auc"]], on=keys, suffixes=("_val", "_test"),
)
display(comparison.round(3))

,model,setting,f1_bot_val,macro_f1_val,ap_val,roc_auc_val,f1_bot_test,macro_f1_test,ap_test,roc_auc_test
0,dummy,default,0.000,0.401,0.332,0.500,0.000,0.401,0.332,0.500
1,isolation_forest,default,0.152,0.458,0.288,0.378,0.137,0.448,0.281,0.368
2,isolation_forest,validation_f1,0.498,0.249,0.288,0.378,0.498,0.249,0.281,0.368
3,one_class_svm,default,0.087,0.436,0.372,0.565,0.094,0.440,0.373,0.564
4,one_class_svm,validation_f1,0.501,0.407,0.372,0.565,0.496,0.403,0.373,0.564
5,gradient_boosting,default,0.695,0.782,0.788,0.862,0.689,0.776,0.789,0.861
6,gradient_boosting,validation_f1,0.709,0.778,0.788,0.862,0.705,0.775,0.789,0.861


## Biểu đồ

Màu dùng bảng đã kiểm tra khả năng phân biệt với người mù màu; Dummy là đường tham chiếu màu xám. Số liệu đầy đủ nằm trong `reports/metrics.csv`.

In [5]:
SURFACE, INK, MUTED, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df"
MODEL_COLORS = {"gradient_boosting": "#2a78d6", "isolation_forest": "#eb6834", "one_class_svm": "#1baf7a"}
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": MUTED, "axes.edgecolor": GRID, "axes.titlecolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "grid.color": GRID,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10, "axes.titlesize": 11,
})
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_figure(fig, filename):
    fig.savefig(FIGURES_DIR / filename, dpi=150, bbox_inches="tight")
    print("Đã lưu:", (FIGURES_DIR / filename).relative_to(project_root).as_posix())

In [6]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
ap_by_model = metrics.groupby("model")["ap"].first()
label_points = {"gradient_boosting": (0.62, 8), "one_class_svm": (0.72, 7), "isolation_forest": (0.72, -14)}
for name, color in MODEL_COLORS.items():
    precision, recall, _ = precision_recall_curve(y_test, model_scores(name, X_test))
    legend_label = f"{name} (AP {ap_by_model[name]:.3f})"
    ax.plot(recall, precision, color=color, linewidth=2, label=legend_label)
    target_recall, offset = label_points[name]
    point = np.argmin(np.abs(recall - target_recall))
    ax.annotate(name, (recall[point], precision[point]), xytext=(0, offset), textcoords="offset points",
                color=INK, ha="left" if name == "gradient_boosting" else "center", fontsize=9)
ax.axhline(bot_share, color=MUTED, linestyle="--", linewidth=1.5, label=f"dummy / ngẫu nhiên (AP {bot_share:.3f})")
ax.set(title="Đường Precision–Recall trên tập test", xlabel="Recall (lớp bot)", ylabel="Precision (lớp bot)",
       xlim=(0, 1), ylim=(0, 1.02))
ax.grid(axis="y", linewidth=0.8)
ax.set_axisbelow(True)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
save_figure(fig, "precision_recall_test.png")
plt.show()

Đã lưu: reports/figures/precision_recall_test.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_20928\658836248.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
panels = [
    ("dummy", "default"), ("isolation_forest", "default"),
    ("one_class_svm", "default"), ("gradient_boosting", "validation_f1"),
]
BLUE_RAMP = ["#cde2fb", "#86b6ef", "#2a78d6", "#104281"]
cmap = plt.matplotlib.colors.LinearSegmentedColormap.from_list("blue", BLUE_RAMP)
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True)
for ax, (name, setting) in zip(axes, panels):
    row = metrics[(metrics["model"] == name) & (metrics["setting"] == setting)].iloc[0]
    matrix = np.array([[row["tn"], row["fp"]], [row["fn"], row["tp"]]])
    ax.imshow(matrix, cmap=cmap, vmin=0, vmax=len(y_test) * (1 - bot_share))
    for (i, j), count in np.ndenumerate(matrix):
        dark_cell = count > 0.5 * len(y_test) * (1 - bot_share)
        ax.text(j, i, f"{count:,}".replace(",", "."), ha="center", va="center",
                color="white" if dark_cell else INK, fontsize=11)
    ax.set_title(f"{name}\n{setting}", fontsize=10)
    ax.set_xticks([0, 1], ["human", "bot"])
    ax.set_yticks([0, 1], ["human", "bot"])
    ax.set_xlabel("Dự đoán")
    ax.tick_params(length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
axes[0].set_ylabel("Nhãn thật")
fig.suptitle("Ma trận nhầm lẫn trên tập test (cùng thang màu)", y=1.06, fontsize=11)
save_figure(fig, "confusion_matrices_test.png")
plt.show()

Đã lưu: reports/figures/confusion_matrices_test.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_20928\2351473747.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Ảnh hưởng của đặc trưng trong Gradient Boosting

- `feature_importances_`: mức giảm impurity trên **train**; không phụ thuộc dữ liệu đánh giá nhưng có thể thiên về đặc trưng nhiều giá trị khác nhau.
- Permutation importance: mức giảm AP trên **validation** khi xáo trộn một đặc trưng (10 lần lặp). Test không được dùng cho phần này.

Importance cho biết mô hình dựa vào đặc trưng nào, không phải quan hệ nhân quả.

In [8]:
gb_model = bundles["gradient_boosting"]["model"]
permutation = permutation_importance(
    gb_model, feature_matrix(X_val), y_val, scoring="average_precision",
    n_repeats=10, random_state=RANDOM_STATE,
)
importance = pd.DataFrame({
    "feature": list(FEATURE_COLUMNS),
    "impurity_importance": gb_model.named_steps["classifier"].feature_importances_,
    "permutation_ap_drop": permutation.importances_mean,
    "permutation_ap_drop_std": permutation.importances_std,
}).sort_values("permutation_ap_drop", ascending=False, ignore_index=True)
importance.to_csv(REPORTS_DIR / "feature_importance.csv", index=False)
display(importance.round(4))

order = importance.iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.2), sharey=True)
panels = (
    ("impurity_importance", None, "Giảm impurity (train)", "{:.3f}"),
    ("permutation_ap_drop", "permutation_ap_drop_std", "Mức giảm AP khi xáo trộn (validation)", "{:.3f}"),
)
for ax, (column, error, title, fmt) in zip(axes, panels):
    bars = ax.barh(order["feature"], order[column], height=0.5, color="#2a78d6",
                   xerr=order[error] if error else None, error_kw={"ecolor": MUTED, "linewidth": 1})
    offset = (order[error].to_numpy() if error else np.zeros(len(order)))
    for bar, extra, value in zip(bars, offset, order[column]):
        ax.text(value + extra + order[column].max() * 0.02, bar.get_y() + bar.get_height() / 2,
                fmt.format(value), va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlim(0, order[column].max() * 1.2)
    ax.grid(axis="x", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.tick_params(axis="y", length=0)
fig.tight_layout()
save_figure(fig, "feature_importance_gb.png")
plt.show()

,feature,impurity_importance,permutation_ap_drop,permutation_ap_drop_std
0,posting_frequency,0.3908,0.2082,0.0054
1,account_age_days,0.2327,0.1533,0.0046
2,follower_following_ratio,0.2208,0.1527,0.0054
3,bio_length,0.1557,0.0773,0.0044


Đã lưu: reports/figures/feature_importance_gb.png


C:\Users\Admin\AppData\Local\Temp\ipykernel_20928\1309867890.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Phân tích dự đoán sai của Gradient Boosting trên test

Dùng thiết lập `validation_f1` đã khóa. Bảng so sánh trung vị đặc trưng theo bốn nhóm TN/FP/FN/TP. `verified`, `default_profile` và tỷ lệ bio rỗng chỉ để diễn giải, **không phải đầu vào mô hình**. Phần này chỉ để hiểu giới hạn của mô hình, không dùng để chỉnh lại mô hình. Không hiển thị tên tài khoản hay nội dung bio.

In [9]:
gb_scores = model_scores("gradient_boosting", X_test)
gb_threshold = bundles["gradient_boosting"]["thresholds"]["validation_f1"]
gb_pred = model_predictions("gradient_boosting", X_test, gb_threshold, gb_scores)
outcome = np.select(
    [(y_test == 0) & (gb_pred == 0), (y_test == 0) & (gb_pred == 1), (y_test == 1) & (gb_pred == 0)],
    ["TN (human đúng)", "FP (human bị gắn bot)", "FN (bot bị bỏ sót)"], default="TP (bot đúng)",
)
context = profiles.loc[test.index]
error_rows = []
for group in ("TN (human đúng)", "FP (human bị gắn bot)", "FN (bot bị bỏ sót)", "TP (bot đúng)"):
    mask = outcome == group
    error_rows.append({
        "group": group, "rows": int(mask.sum()), "median_bot_probability": np.median(gb_scores[mask]),
        **X_test[mask].astype("float64").median().to_dict(),
        "verified_share": context.loc[mask, "verified"].eq("True").mean(),
        "default_profile_share": context.loc[mask, "default_profile"].eq("True").mean(),
        "empty_bio_share": X_test.loc[mask, "bio_length"].eq(0).mean(),
    })
error_analysis = pd.DataFrame(error_rows)
error_analysis.to_csv(REPORTS_DIR / "error_analysis.csv", index=False)
display(error_analysis.round(3))

fp_mask, fn_mask = outcome == "FP (human bị gắn bot)", outcome == "FN (bot bị bỏ sót)"
print(f"FP tự tin (xác suất ≥ 0,8): {int((gb_scores[fp_mask] >= 0.8).sum())} / {int(fp_mask.sum())}")
print(f"FN tự tin (xác suất ≤ 0,2): {int((gb_scores[fn_mask] <= 0.2).sum())} / {int(fn_mask.sum())}")

,group,rows,median_bot_probability,follower_following_ratio,posting_frequency,account_age_days,bio_length,verified_share,default_profile_share,empty_bio_share
0,TN (human đúng),4144,0.117,1.908,3.004,3512.0,89.0,0.341,0.281,0.054
1,FP (human bị gắn bot),859,0.534,0.809,0.815,2901.0,21.0,0.075,0.503,0.390
2,FN (bot bị bỏ sót),664,0.198,1.047,2.969,3183.0,77.0,0.068,0.342,0.102
3,TP (bot đúng),1821,0.757,1.000,0.607,2829.0,16.0,0.009,0.726,0.442


FP tự tin (xác suất ≥ 0,8): 71 / 859
FN tự tin (xác suất ≤ 0,2): 338 / 664


## Kết luận và giới hạn

Các bảng và nhận xét dưới đây được tạo từ lần chạy hiện tại, cùng `run_id` và hash model trong `reports/evaluation_run.json`. Không dùng lại số liệu của model tạm để mô tả model chính thức.

- Đối chiếu cả AP, F1 bot và Macro-F1; accuracy cao của Dummy không có nghĩa là phát hiện được bot.
- So sánh detector với mốc gắn nhãn tất cả là bot; giữ rõ ngưỡng mặc định và ngưỡng validation.
- Feature importance được tính trên train/validation; không thể suy ra quan hệ nhân quả.
- Chênh lệch validation/test trên một split không đủ để kết luận không có overfitting.

**Giới hạn:**

- Chỉ dùng bốn đặc trưng hồ sơ, không có nội dung bài đăng hay mạng lưới tương tác. Các cột có sẵn như `verified` và `default_profile` chưa được dùng; bảng phân tích lỗi cho thấy chúng khác nhau giữa các nhóm, nhưng chưa kiểm chứng việc thêm chúng có cải thiện mô hình hay không.
- `posting_frequency` là trung bình cả vòng đời và `account_age_days` là giá trị lịch sử của nguồn, không phản ánh hoạt động gần đây.
- Dữ liệu là một bộ đã gán nhãn sẵn và được chia ngẫu nhiên, nên kết quả không chứng minh mô hình hoạt động tốt với tài khoản mới hoặc bot đổi hành vi theo thời gian.
- Kết quả là ước lượng điểm trên một lần chia, chưa có khoảng tin cậy.

- Dự đoán có false positive/false negative; điểm bot không đủ để kết luận tài khoản giả mạo hay lừa đảo. Lịch sử quan sát test hạn chế mức độc lập của đánh giá.


In [10]:
from IPython.display import Markdown

selected = metrics[metrics["setting"].eq("validation_f1") | metrics["model"].eq("dummy")]
display(selected[["model", "setting", "precision_bot", "recall_bot", "f1_bot", "macro_f1", "ap", "roc_auc", "accuracy"]].round(4))
gb = selected[selected["model"].eq("gradient_boosting")].iloc[0]
display(Markdown(
    f"Gradient Boosting chính thức trên test: **F1 bot {gb['f1_bot']:.4f}**, "
    f"**AP {gb['ap']:.4f}**, **ROC-AUC {gb['roc_auc']:.4f}**. "
    f"Đặc trưng có permutation importance lớn nhất: **{importance.iloc[0]['feature']}** "
    f"(AP giảm {importance.iloc[0]['permutation_ap_drop']:.4f} trên validation). "
    "Cấu hình và ngưỡng giữ nguyên từ notebook 03."
))


,model,setting,precision_bot,recall_bot,f1_bot,macro_f1,ap,roc_auc,accuracy
0,dummy,default,0.0000,0.0000,0.0000,0.4005,0.3319,0.5000,0.6681
2,isolation_forest,validation_f1,0.3316,0.9984,0.4978,0.2493,0.2809,0.3685,0.3316
4,one_class_svm,validation_f1,0.3479,0.8636,0.4960,0.4032,0.3731,0.5643,0.4176
6,gradient_boosting,validation_f1,0.6795,0.7328,0.7051,0.7749,0.7889,0.8613,0.7966


Gradient Boosting chính thức trên test: **F1 bot 0.7051**, **AP 0.7889**, **ROC-AUC 0.8613**. Đặc trưng có permutation importance lớn nhất: **posting_frequency** (AP giảm 0.2082 trên validation). Cấu hình và ngưỡng giữ nguyên từ notebook 03.